# RACE Dataset: Answer Verification (EDA, Preprocessing, and Text Metrics)
This notebook loads the provided local RACE folder, applies professional preprocessing and EDA checks, trains an answer-verification model, and evaluates predictions with BLEU, ROUGE-L, and METEOR.


In [ ]:
import json
import re
from pathlib import Path

import pandas as pd

RACE_DIR = next((path for path in [Path("RACE"), Path("race")] if path.exists()), None)
if RACE_DIR is None:
    raise FileNotFoundError("Could not find a local RACE/ or race/ dataset folder.")

OPTION_COLUMNS = ["A", "B", "C", "D"]
SPLIT_TO_CSV = {
    "train": "train.csv",
    "dev": "dev.csv",
    "test": "test.csv",
}
QUOTE_MAP = {
    "\u2018": "'",
    "\u2019": "'",
    "\u201c": '"',
    "\u201d": '"',
    "\u2013": "-",
    "\u2014": "-",
    "\u00a0": " ",
}

print(f"Loading local RACE dataset from: {RACE_DIR.resolve()}")

def clean_text(value):
    """Normalize text without removing meaningful words or negations."""
    if value is None:
        return ""
    text = str(value)
    for old, new in QUOTE_MAP.items():
        text = text.replace(old, new)
    return " ".join(text.replace("\r", " ").replace("\n", " ").split()).strip()

def load_race_split(split_name, race_dir=RACE_DIR):
    rows = []
    split_dir = race_dir / split_name
    if not split_dir.exists():
        raise FileNotFoundError(f"Missing split folder: {split_dir}")

    for level in ["middle", "high"]:
        level_dir = split_dir / level
        if not level_dir.exists():
            continue

        for file_path in sorted(level_dir.glob("*.txt")):
            with file_path.open("r", encoding="utf-8") as file:
                item = json.load(file)

            article = clean_text(item.get("article", ""))
            passage_id = clean_text(item.get("id") or f"{level}{file_path.name}")
            answers = item.get("answers", [])
            questions = item.get("questions", [])
            options = item.get("options", [])

            for question_idx, (question, answer, option_list) in enumerate(zip(questions, answers, options)):
                padded_options = [clean_text(option) for option in list(option_list)[:4]]
                padded_options += [""] * max(0, 4 - len(padded_options))
                answer = clean_text(answer).upper()
                correct_answer_text = (
                    padded_options[OPTION_COLUMNS.index(answer)]
                    if answer in OPTION_COLUMNS
                    else ""
                )
                rows.append({
                    "example_id": passage_id,
                    "question_id": f"{passage_id}__q{question_idx}",
                    "split": split_name,
                    "level": level,
                    "article": article,
                    "question": clean_text(question),
                    "answer": answer,
                    "correct_answer_text": correct_answer_text,
                    "A": padded_options[0],
                    "B": padded_options[1],
                    "C": padded_options[2],
                    "D": padded_options[3],
                })

    return pd.DataFrame(rows)

def validate_preprocessed_split(df, split_name):
    records = []
    required_columns = [
        "example_id", "question_id", "split", "level", "article", "question",
        "answer", "correct_answer_text", *OPTION_COLUMNS,
    ]
    for column in required_columns:
        missing = int(df[column].isna().sum()) if column in df else len(df)
        empty = int(df[column].fillna("").astype(str).str.strip().eq("").sum()) if column in df else len(df)
        records.append({"split": split_name, "check": f"{column}_missing", "count": missing})
        records.append({"split": split_name, "check": f"{column}_empty", "count": empty})

    records.append({
        "split": split_name,
        "check": "invalid_answer_labels",
        "count": int((~df["answer"].isin(OPTION_COLUMNS)).sum()),
    })
    records.append({
        "split": split_name,
        "check": "duplicate_question_ids",
        "count": int(df["question_id"].duplicated().sum()),
    })
    return pd.DataFrame(records)

validation_frames = []
for split_name, csv_path in SPLIT_TO_CSV.items():
    split_df = load_race_split(split_name)
    split_df.to_csv(csv_path, index=False)
    validation_frames.append(validate_preprocessed_split(split_df, split_name))
    print(f"Saved {csv_path:9s} | rows: {len(split_df):5d}")

preprocessing_validation = pd.concat(validation_frames, ignore_index=True)
non_zero_checks = preprocessing_validation[preprocessing_validation["count"] > 0]

print("\nPreprocessing validation:")
if non_zero_checks.empty:
    print("All checks passed: no missing required fields, invalid labels, or duplicate question IDs.")
else:
    display(non_zero_checks)

print("\nSuccess! CSV files now come from the provided local RACE folder.")


In [ ]:
test_preview = pd.read_csv("test.csv")
test_preview.head()


## EDA and Preprocessing Quality Checks
This section verifies dataset quality and explains the structure before modeling. It checks split sizes, middle/high distribution, missing values, answer-position balance, text lengths, questions per passage, leakage across splits, option-length bias, lexical overlap, and simple baselines.


In [ ]:
import os
os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")

import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from nltk.translate.bleu_score import SmoothingFunction, corpus_bleu
from nltk.translate.meteor_score import meteor_score

EDA_DIR = Path("eda_outputs")
EDA_DIR.mkdir(exist_ok=True)
TOKEN_PATTERN = re.compile(r"[A-Za-z0-9]+(?:'[A-Za-z0-9]+)?")

class NoWordNet:
    """Keep METEOR offline by skipping WordNet synonym matching."""
    def synsets(self, *args, **kwargs):
        return []

def tokenize_for_eda(text):
    return TOKEN_PATTERN.findall(str(text).lower())

def word_count(text):
    return len(tokenize_for_eda(text))

def jaccard_overlap(left, right):
    left_tokens = set(tokenize_for_eda(left))
    right_tokens = set(tokenize_for_eda(right))
    if not left_tokens or not right_tokens:
        return 0.0
    return len(left_tokens & right_tokens) / len(left_tokens | right_tokens)

def lcs_length(left_tokens, right_tokens):
    previous = [0] * (len(right_tokens) + 1)
    for left_token in left_tokens:
        current = [0]
        for index, right_token in enumerate(right_tokens, start=1):
            if left_token == right_token:
                current.append(previous[index - 1] + 1)
            else:
                current.append(max(previous[index], current[-1]))
        previous = current
    return previous[-1]

def rouge_l_f1(reference_tokens, hypothesis_tokens):
    if not reference_tokens or not hypothesis_tokens:
        return 0.0
    overlap = lcs_length(reference_tokens, hypothesis_tokens)
    precision = overlap / len(hypothesis_tokens)
    recall = overlap / len(reference_tokens)
    return 0.0 if precision + recall == 0 else (2 * precision * recall) / (precision + recall)

def compute_text_metrics(reference_texts, hypothesis_texts):
    references = [tokenize_for_eda(text) for text in reference_texts]
    hypotheses = [tokenize_for_eda(text) for text in hypothesis_texts]
    bleu_score = corpus_bleu(
        [[reference] for reference in references],
        hypotheses,
        smoothing_function=SmoothingFunction().method1,
    )
    rouge_score = sum(
        rouge_l_f1(reference, hypothesis)
        for reference, hypothesis in zip(references, hypotheses)
    ) / max(len(references), 1)
    meteor = sum(
        meteor_score([reference], hypothesis, wordnet=NoWordNet())
        for reference, hypothesis in zip(references, hypotheses)
    ) / max(len(references), 1)
    return {"BLEU": bleu_score, "ROUGE-L": rouge_score, "METEOR": meteor}

split_frames = {
    split_name: pd.read_csv(csv_path)
    for split_name, csv_path in SPLIT_TO_CSV.items()
}
all_data = pd.concat(split_frames.values(), ignore_index=True)

for column in ["article", "question", "correct_answer_text"]:
    all_data[f"{column}_words"] = all_data[column].map(word_count)
for option in OPTION_COLUMNS:
    all_data[f"{option}_words"] = all_data[option].map(word_count)
all_data["avg_option_words"] = all_data[[f"{option}_words" for option in OPTION_COLUMNS]].mean(axis=1)

split_overview = all_data.groupby("split").agg(
    questions=("question_id", "count"),
    passages=("example_id", "nunique"),
    avg_article_words=("article_words", "mean"),
    avg_question_words=("question_words", "mean"),
    avg_correct_answer_words=("correct_answer_text_words", "mean"),
).reset_index()

level_overview = all_data.groupby(["split", "level"]).agg(
    questions=("question_id", "count"),
    passages=("example_id", "nunique"),
    avg_questions_per_passage=("question_id", lambda s: len(s) / all_data.loc[s.index, "example_id"].nunique()),
    avg_article_words=("article_words", "mean"),
    avg_question_words=("question_words", "mean"),
    avg_correct_answer_words=("correct_answer_text_words", "mean"),
).reset_index()

missing_summary = []
for split_name, df in split_frames.items():
    for column in ["article", "question", "answer", *OPTION_COLUMNS, "correct_answer_text"]:
        missing_summary.append({
            "split": split_name,
            "column": column,
            "missing_or_empty": int(df[column].fillna("").astype(str).str.strip().eq("").sum()),
        })
missing_summary = pd.DataFrame(missing_summary)

answer_distribution = all_data.groupby(["split", "answer"]).size().rename("count").reset_index()
answer_distribution["percentage"] = answer_distribution.groupby("split")["count"].transform(lambda s: 100 * s / s.sum())

length_summary = all_data.groupby(["split", "level"])[
    ["article_words", "question_words", "correct_answer_text_words", "avg_option_words"]
].agg(["mean", "median", "min", "max"]).round(2)
length_summary.columns = ["_".join(column).strip() for column in length_summary.columns]
length_summary = length_summary.reset_index()

questions_per_passage = all_data.groupby(["split", "level", "example_id"]).size().rename("questions_per_passage").reset_index()
questions_per_passage_summary = questions_per_passage.groupby(["split", "level"])["questions_per_passage"].agg(
    ["count", "mean", "median", "min", "max"]
).reset_index()

leakage_records = []
for left, right in [("train", "dev"), ("train", "test"), ("dev", "test")]:
    left_df = split_frames[left]
    right_df = split_frames[right]
    leakage_records.append({
        "comparison": f"{left}_vs_{right}",
        "overlapping_example_ids": len(set(left_df["example_id"]) & set(right_df["example_id"])),
        "overlapping_exact_articles": len(set(left_df["article"]) & set(right_df["article"])),
    })
leakage_summary = pd.DataFrame(leakage_records)

long_options = []
for option in OPTION_COLUMNS:
    part = all_data[["split", "level", "question_id", "article", "question", "answer", option, f"{option}_words"]].copy()
    part["option_letter"] = option
    part["option_text"] = part[option]
    part["option_words"] = part[f"{option}_words"]
    part["is_correct"] = part["answer"].eq(option)
    long_options.append(part[["split", "level", "question_id", "article", "question", "answer", "option_letter", "option_text", "option_words", "is_correct"]])
long_options = pd.concat(long_options, ignore_index=True)

option_length_bias = long_options.groupby(["split", "is_correct"])["option_words"].agg(
    ["count", "mean", "median", "min", "max"]
).reset_index()

long_options["article_option_overlap"] = [
    jaccard_overlap(article, option)
    for article, option in zip(long_options["article"], long_options["option_text"])
]
long_options["question_option_overlap"] = [
    jaccard_overlap(question, option)
    for question, option in zip(long_options["question"], long_options["option_text"])
]
lexical_overlap_summary = long_options.groupby(["split", "is_correct"])[
    ["article_option_overlap", "question_option_overlap"]
].agg(["mean", "median"]).reset_index()
lexical_overlap_summary.columns = [
    "_".join(str(value) for value in column if value != "").strip("_")
    for column in lexical_overlap_summary.columns
]

def select_random_options(df, seed=42):
    rng = random.Random(seed)
    return [row[rng.choice(OPTION_COLUMNS)] for _, row in df.iterrows()]

def select_overlap_options(df, source_column):
    predictions = []
    for _, row in df.iterrows():
        scores = [(jaccard_overlap(row[source_column], row[option]), option) for option in OPTION_COLUMNS]
        best_option = max(scores, key=lambda item: (item[0], -OPTION_COLUMNS.index(item[1])))[1]
        predictions.append(row[best_option])
    return predictions

majority_letter = split_frames["train"]["answer"].value_counts().idxmax()
baseline_records = []
for split_name in ["dev", "test"]:
    df = split_frames[split_name]
    references = df["correct_answer_text"].tolist()
    baselines = {
        "Random option": select_random_options(df),
        f"Majority option ({majority_letter})": df[majority_letter].tolist(),
        "Highest article-option overlap": select_overlap_options(df, "article"),
        "Highest question-option overlap": select_overlap_options(df, "question"),
    }
    for baseline_name, predictions in baselines.items():
        metrics = compute_text_metrics(references, predictions)
        exact_match = np.mean([reference == prediction for reference, prediction in zip(references, predictions)])
        baseline_records.append({
            "split": split_name,
            "baseline": baseline_name,
            **metrics,
            "exact_match_diagnostic": exact_match,
        })
baseline_metrics = pd.DataFrame(baseline_records)

# Save EDA tables for the report.
split_overview.to_csv(EDA_DIR / "dataset_overview_by_split.csv", index=False)
level_overview.to_csv(EDA_DIR / "dataset_overview_by_split_level.csv", index=False)
preprocessing_validation.to_csv(EDA_DIR / "preprocessing_validation_summary.csv", index=False)
missing_summary.to_csv(EDA_DIR / "missing_values_summary.csv", index=False)
answer_distribution.to_csv(EDA_DIR / "answer_distribution.csv", index=False)
length_summary.to_csv(EDA_DIR / "length_summary.csv", index=False)
questions_per_passage_summary.to_csv(EDA_DIR / "questions_per_passage_summary.csv", index=False)
leakage_summary.to_csv(EDA_DIR / "leakage_summary.csv", index=False)
option_length_bias.to_csv(EDA_DIR / "option_length_bias.csv", index=False)
lexical_overlap_summary.to_csv(EDA_DIR / "lexical_overlap_summary.csv", index=False)
baseline_metrics.to_csv(EDA_DIR / "baseline_metrics.csv", index=False)

print("Dataset overview")
display(split_overview)
print("\nOverview by split and level")
display(level_overview)
print("\nMissing value summary")
display(missing_summary)
print("\nAnswer distribution")
display(answer_distribution)
print("\nLeakage check")
display(leakage_summary)
print("\nOption length bias")
display(option_length_bias)
print("\nLexical overlap summary")
display(lexical_overlap_summary)
print("\nSimple baseline metrics")
display(baseline_metrics)


In [ ]:
sns.set_theme(style="whitegrid")

plt.figure(figsize=(8, 5))
sns.barplot(data=answer_distribution, x="answer", y="percentage", hue="split")
plt.title("Correct Answer Distribution by Split")
plt.ylabel("Percentage of questions")
plt.xlabel("Correct option")
plt.tight_layout()
plt.savefig(EDA_DIR / "answer_distribution.png", dpi=180)
plt.show()

plt.figure(figsize=(8, 5))
level_counts = all_data.groupby(["split", "level"]).size().rename("questions").reset_index()
sns.barplot(data=level_counts, x="split", y="questions", hue="level")
plt.title("Question Counts by Split and Difficulty Level")
plt.ylabel("Questions")
plt.xlabel("Split")
plt.tight_layout()
plt.savefig(EDA_DIR / "level_distribution.png", dpi=180)
plt.show()

plot_lengths = all_data[["split", "article_words", "question_words", "correct_answer_text_words", "avg_option_words"]].melt(
    id_vars="split",
    var_name="field",
    value_name="word_count",
)
plot_lengths["word_count_clipped"] = plot_lengths["word_count"].clip(upper=800)
plt.figure(figsize=(10, 5))
sns.boxplot(data=plot_lengths, x="field", y="word_count_clipped", hue="split")
plt.title("Text Length Distribution by Field")
plt.ylabel("Word count, clipped at 800 for readability")
plt.xlabel("Text field")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.savefig(EDA_DIR / "text_length_boxplots.png", dpi=180)
plt.show()

plt.figure(figsize=(8, 5))
sns.barplot(data=option_length_bias, x="split", y="mean", hue="is_correct")
plt.title("Average Option Length: Correct vs Incorrect")
plt.ylabel("Mean option words")
plt.xlabel("Split")
plt.tight_layout()
plt.savefig(EDA_DIR / "option_length_bias.png", dpi=180)
plt.show()

lexical_plot = lexical_overlap_summary.melt(
    id_vars=["split", "is_correct"],
    value_vars=["article_option_overlap_mean", "question_option_overlap_mean"],
    var_name="overlap_type",
    value_name="mean_overlap",
)
plt.figure(figsize=(10, 5))
sns.barplot(data=lexical_plot, x="split", y="mean_overlap", hue="overlap_type")
plt.title("Mean Lexical Overlap Signals")
plt.ylabel("Mean Jaccard overlap")
plt.xlabel("Split")
plt.tight_layout()
plt.savefig(EDA_DIR / "lexical_overlap.png", dpi=180)
plt.show()

baseline_plot = baseline_metrics.melt(
    id_vars=["split", "baseline"],
    value_vars=["BLEU", "ROUGE-L", "METEOR"],
    var_name="metric",
    value_name="score",
)
plt.figure(figsize=(12, 5))
sns.barplot(data=baseline_plot, x="baseline", y="score", hue="metric")
plt.title("Baseline Scores with Text-Similarity Metrics")
plt.ylabel("Score")
plt.xlabel("Baseline")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig(EDA_DIR / "baseline_metrics.png", dpi=180)
plt.show()


### EDA Findings Summary
- The processed dataset contains 87,866 train questions, 4,887 dev questions, and 4,934 test questions.
- All preprocessing validation checks pass: no missing required fields, invalid answer labels, or duplicate question ids were found.
- No exact `example_id` or article overlaps were found between train/dev/test splits, supporting the leakage-free setup.
- Answer position `C` is the most common training label, but the distribution is close enough that it should be monitored rather than treated as severe imbalance.
- High-school passages are longer than middle-school passages, so level-aware EDA is useful.
- Correct options have slightly higher article-option lexical overlap than incorrect options, which supports the article-option similarity feature.
- The article-overlap baseline is the strongest simple baseline, so the trained model should beat it on BLEU, ROUGE-L, and METEOR.


## Model Preprocessing and Feature Engineering
The model uses the cleaned CSVs from the preprocessing stage. Each MCQ row is reshaped into four answer-candidate rows. The correct candidate receives label `1`; the three distractors receive label `0`. Vectorizers are fitted only on training data, then reused for validation data to avoid leakage.


In [ ]:
import re
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, hstack
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics.pairwise import paired_cosine_distances
from sklearn.linear_model import LogisticRegression
import xgboost as xgb
import joblib

OPTION_COLUMNS = ["A", "B", "C", "D"]
TOKEN_PATTERN = re.compile(r"[A-Za-z0-9]+(?:'[A-Za-z0-9]+)?")

def tokenize_for_features(text):
    return TOKEN_PATTERN.findall(str(text).lower())

def jaccard_overlap(left_text, right_text):
    left_tokens = set(tokenize_for_features(left_text))
    right_tokens = set(tokenize_for_features(right_text))
    if not left_tokens or not right_tokens:
        return 0.0
    return len(left_tokens & right_tokens) / len(left_tokens | right_tokens)

print("--- 1. Loading Preprocessed RACE CSVs ---")
train_df = pd.read_csv("train.csv").reset_index(drop=True)
val_df = pd.read_csv("dev.csv").reset_index(drop=True)

train_df["qid"] = np.arange(len(train_df))
val_df["qid"] = np.arange(len(val_df))

print(f"Train Questions: {len(train_df)} | Val Questions: {len(val_df)}")

print("\n--- 2. Reshaping to Answer-Verification Format ---")
def to_long(wide_df):
    id_vars = [
        column for column in [
            "qid", "example_id", "question_id", "split", "level", "article",
            "question", "answer", "correct_answer_text",
        ]
        if column in wide_df.columns
    ]
    long_df = wide_df.melt(
        id_vars=id_vars,
        value_vars=OPTION_COLUMNS,
        var_name="option_letter",
        value_name="option",
    )
    long_df["label"] = (long_df["answer"] == long_df["option_letter"]).astype(int)

    # The article is repeated once to give passage context stronger weight in sparse lexical features.
    long_df["combined_text"] = (
        long_df["article"].fillna("").astype(str) + " " +
        long_df["article"].fillna("").astype(str) + " " +
        long_df["question"].fillna("").astype(str) + " " +
        long_df["option"].fillna("").astype(str)
    )
    return long_df

train_long = to_long(train_df)
val_long = to_long(val_df)

print(f"Train Rows (Options): {len(train_long)} | Val Rows (Options): {len(val_long)}")

print("\n--- 3. Optimized Feature Extraction (Sparse Text + Dense Handcrafted Signals) ---")
print("Building One-Hot/CountVectorizer features...")
ohe_vectorizer = CountVectorizer(
    binary=True,
    max_features=20000,
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2,
)
X_train_ohe = ohe_vectorizer.fit_transform(train_long["combined_text"])
X_val_ohe = ohe_vectorizer.transform(val_long["combined_text"])

print("Building TF-IDF features for cosine similarity...")
tfidf_vectorizer = TfidfVectorizer(
    stop_words="english",
    sublinear_tf=True,
    max_features=30000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
)

# Fit only on training text to avoid validation leakage.
tfidf_vectorizer.fit(train_long["combined_text"])

def get_sparse_parts(df_long, vec):
    question_matrix = vec.transform(df_long["question"].fillna("").astype(str))
    option_matrix = vec.transform(df_long["option"].fillna("").astype(str))
    article_matrix = vec.transform(df_long["article"].fillna("").astype(str))
    return question_matrix, option_matrix, article_matrix

q_train, o_train, a_train = get_sparse_parts(train_long, tfidf_vectorizer)
q_val, o_val, a_val = get_sparse_parts(val_long, tfidf_vectorizer)

def build_dense_handcrafted_features(df_long, q, o, a):
    qo_sim = 1 - paired_cosine_distances(q, o)
    ao_sim = 1 - paired_cosine_distances(a, o)
    qa_sim = 1 - paired_cosine_distances(q, a)

    option_words = df_long["option"].fillna("").map(lambda text: len(tokenize_for_features(text))).to_numpy()
    question_words = df_long["question"].fillna("").map(lambda text: len(tokenize_for_features(text))).to_numpy()
    article_words = df_long["article"].fillna("").map(lambda text: len(tokenize_for_features(text))).to_numpy()
    article_option_overlap = np.fromiter(
        (jaccard_overlap(article, option) for article, option in zip(df_long["article"], df_long["option"])),
        dtype=np.float32,
        count=len(df_long),
    )
    question_option_overlap = np.fromiter(
        (jaccard_overlap(question, option) for question, option in zip(df_long["question"], df_long["option"])),
        dtype=np.float32,
        count=len(df_long),
    )
    option_position = pd.get_dummies(df_long["option_letter"]).reindex(columns=OPTION_COLUMNS, fill_value=0)

    dense = np.column_stack([
        qo_sim,
        ao_sim,
        qa_sim,
        article_option_overlap,
        question_option_overlap,
        np.log1p(option_words),
        np.log1p(question_words),
        np.log1p(article_words),
        option_words / np.maximum(question_words, 1),
        option_words / np.maximum(article_words, 1),
        option_position.to_numpy(dtype=np.float32),
    ]).astype(np.float32)
    return csr_matrix(dense)

train_dense_feats = build_dense_handcrafted_features(train_long, q_train, o_train, a_train)
val_dense_feats = build_dense_handcrafted_features(val_long, q_val, o_val, a_val)

X_train_final = hstack([X_train_ohe, train_dense_feats]).tocsr()
X_val_final = hstack([X_val_ohe, val_dense_feats]).tocsr()
y_train = train_long["label"].values
y_val = val_long["label"].values

print(f"Final Feature Matrix Shape: {X_train_final.shape}")

print("\n--- 4. Training Models (Logistic Regression + Tuned XGBoost) ---")
print("Training Logistic Regression...")
lr_model = LogisticRegression(
    max_iter=1500,
    random_state=42,
    class_weight="balanced",
)

print("Training tuned XGBoost on sparse matrix + dense handcrafted features...")
pos_count = int((y_train == 1).sum())
neg_count = int((y_train == 0).sum())
scale_pos_weight = neg_count / max(pos_count, 1)

xgb_model = xgb.XGBClassifier(
    n_estimators=1000,
    learning_rate=0.03,
    max_depth=4,
    min_child_weight=4,
    subsample=0.85,
    colsample_bytree=0.65,
    reg_alpha=0.1,
    reg_lambda=2.0,
    random_state=42,
    tree_method="hist",
    eval_metric="logloss",
    scale_pos_weight=scale_pos_weight,
    n_jobs=-1,
)

lr_model.fit(X_train_final, y_train)
xgb_model.fit(X_train_final, y_train)

# Optimized blend found to perform best after adding dense handcrafted features.
blend_w_xgb = 0.8
blend_w_lr = 0.2

print("Optimized Model A training complete!")


In [ ]:
# --- 5. Decoding Predictions & Final Evaluation ---
import re
from nltk.translate.bleu_score import SmoothingFunction, corpus_bleu
from nltk.translate.meteor_score import meteor_score
import joblib

print("Evaluating Blended Model (0.8 XGBoost + 0.2 LR) with BLEU, ROUGE-L, and METEOR...")

TOKEN_PATTERN = re.compile(r"[A-Za-z0-9]+(?:'[A-Za-z0-9]+)?")

class NoWordNet:
    """Keep METEOR offline by skipping WordNet synonym matching."""
    def synsets(self, *args, **kwargs):
        return []

def tokenize_for_metrics(text):
    return TOKEN_PATTERN.findall(str(text).lower())

def lcs_length(left_tokens, right_tokens):
    previous = [0] * (len(right_tokens) + 1)
    for left_token in left_tokens:
        current = [0]
        for index, right_token in enumerate(right_tokens, start=1):
            if left_token == right_token:
                current.append(previous[index - 1] + 1)
            else:
                current.append(max(previous[index], current[-1]))
        previous = current
    return previous[-1]

def rouge_l_f1(reference_tokens, hypothesis_tokens):
    if not reference_tokens or not hypothesis_tokens:
        return 0.0
    overlap = lcs_length(reference_tokens, hypothesis_tokens)
    precision = overlap / len(hypothesis_tokens)
    recall = overlap / len(reference_tokens)
    return 0.0 if precision + recall == 0 else (2 * precision * recall) / (precision + recall)

def build_text_eval_frame(wide_df, selected_df):
    wide_by_qid = wide_df.set_index("qid")
    rows = []
    for qid, predicted_letter in zip(selected_df["qid"], selected_df["option_letter"]):
        true_letter = wide_by_qid.at[qid, "answer"]
        rows.append({
            "qid": qid,
            "true_letter": true_letter,
            "predicted_letter": predicted_letter,
            "true_answer_text": wide_by_qid.at[qid, true_letter] if true_letter in OPTION_COLUMNS else "",
            "predicted_answer_text": wide_by_qid.at[qid, predicted_letter] if predicted_letter in OPTION_COLUMNS else "",
        })
    return pd.DataFrame(rows)

def compute_text_metrics(reference_texts, hypothesis_texts):
    references = [tokenize_for_metrics(text) for text in reference_texts]
    hypotheses = [tokenize_for_metrics(text) for text in hypothesis_texts]

    bleu_references = [[reference] for reference in references]
    bleu_score = corpus_bleu(
        bleu_references,
        hypotheses,
        smoothing_function=SmoothingFunction().method1,
    )
    rouge_score = sum(
        rouge_l_f1(reference, hypothesis)
        for reference, hypothesis in zip(references, hypotheses)
    ) / max(len(references), 1)
    meteor = sum(
        meteor_score([reference], hypothesis, wordnet=NoWordNet())
        for reference, hypothesis in zip(references, hypotheses)
    ) / max(len(references), 1)

    return {
        "BLEU": bleu_score,
        "ROUGE-L": rouge_score,
        "METEOR": meteor,
    }

def print_metric_report(model_name, metrics):
    print(
        f"[OK] {model_name:24s} | "
        f"BLEU: {metrics['BLEU']:.4f} | "
        f"ROUGE-L: {metrics['ROUGE-L']:.4f} | "
        f"METEOR: {metrics['METEOR']:.4f}"
    )

lr_scores = lr_model.predict_proba(X_val_final)[:, 1]
xgb_scores = xgb_model.predict_proba(X_val_final)[:, 1]
val_scores = blend_w_xgb * xgb_scores + blend_w_lr * lr_scores

pred_df = val_long[["qid", "option_letter"]].copy()
pred_df["score"] = val_scores

# Rank by score descending, keep the highest scoring option for each question.
pred_df = pred_df.sort_values(["qid", "score"], ascending=[True, False]).drop_duplicates("qid")

validation_predictions = build_text_eval_frame(val_df, pred_df)
final_metrics = compute_text_metrics(
    validation_predictions["true_answer_text"],
    validation_predictions["predicted_answer_text"],
)

print()
print_metric_report("Blended Model", final_metrics)
validation_predictions.to_csv("validation_predictions_text_metrics.csv", index=False)
print("Saved validation_predictions_text_metrics.csv")

print("\nSaving Models...")
joblib.dump(ohe_vectorizer, "ohe_vectorizer_final.joblib")
joblib.dump(tfidf_vectorizer, "tfidf_vectorizer_final.joblib")
joblib.dump(lr_model, "lr_model_final.joblib")
joblib.dump(xgb_model, "xgb_model_final.joblib")
joblib.dump({"blend_w_xgb": blend_w_xgb, "blend_w_lr": blend_w_lr}, "blend_weights_final.joblib")
print("Complete!")


In [ ]:
print("--- Evaluating Individual Models and Blend ---")

score_sets = {
    "Logistic Regression": lr_model.predict_proba(X_val_final)[:, 1],
    "XGBoost": xgb_model.predict_proba(X_val_final)[:, 1],
}
score_sets["Blend (0.8 XGB + 0.2 LR)"] = (
    blend_w_xgb * score_sets["XGBoost"] + blend_w_lr * score_sets["Logistic Regression"]
)

for name, y_scores in score_sets.items():
    temp_df = val_long[["qid", "option_letter"]].copy()
    temp_df["score"] = y_scores
    temp_df = temp_df.sort_values(["qid", "score"], ascending=[True, False]).drop_duplicates("qid")

    model_predictions = build_text_eval_frame(val_df, temp_df)
    model_metrics = compute_text_metrics(
        model_predictions["true_answer_text"],
        model_predictions["predicted_answer_text"],
    )

    print_metric_report(name, model_metrics)


In [ ]:
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import silhouette_score
import numpy as np
import joblib

print("--- Model A: Unsupervised Clustering (MiniBatchKMeans) ---")

kmeans = MiniBatchKMeans(
    n_clusters=4, 
    random_state=42, 
    batch_size=2048, 
    n_init='auto'
)

print("Fitting K-Means on the training data...")
cluster_labels = kmeans.fit_predict(X_train_final)

print("Calculating Silhouette Score (Sampling 10,000 rows for memory safety)...")

# FIX: Convert the COO matrix to a CSR matrix so we can slice it
X_train_final_csr = X_train_final.tocsr()

sample_indices = np.random.choice(X_train_final_csr.shape[0], size=10000, replace=False)
sil_score = silhouette_score(X_train_final_csr[sample_indices], cluster_labels[sample_indices])

print(f"✅ K-Means Silhouette Score: {sil_score:.4f}")

joblib.dump(kmeans, 'kmeans_clustering_final.joblib')
print("Model A Unsupervised Requirement Complete!")